In [1]:
# final_rare_event_ranker.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from typing import Dict, Tuple, Sequence, Union
from resolve.utilities import utilities as utils
import yaml
import random
import os
import h5py
from pathlib import Path
from sklearn.utils import shuffle

In [2]:



# ------------------------------
# Repro
# ------------------------------
from torch import logit


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _get_hdf5_files(path_to_files, config_file):
        return sorted(str(p) for p in path_to_files.glob(f"*.{config_file['simulation_settings']['file_format']}"))

# ------------------------------
# Data utilities + file pipeline
# ------------------------------
def _read_in_from_file(file_path: str, parameter_config: Dict) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Read (theta, phi, y) from HDF5 or CSV and return tensors (x, y).
    - x = [theta | phi] concatenated along last dimension, float32
    - y shaped (N, 1) float32

    parameter_config can provide either 'key'+'selected_indices' (for HDF5) or 'selected_labels' (for CSV):
      'phi':   {'key' or 'selected_labels', 'selected_indices'}
      'theta': {'key' or 'selected_labels', 'selected_indices'}
      'target':{'key' or 'selected_labels', 'selected_indices'}
    """
    if file_path.endswith(('.h5', '.hdf5')):
        with h5py.File(file_path, 'r') as hdf:
            # φ
            phi = hdf[parameter_config['phi']['key']][:, parameter_config['phi']['selected_indices']]
            # θ
            theta = hdf[parameter_config['theta']['key']]
            if len(parameter_config['theta']['selected_indices']) != 0:
                if theta.ndim == 1:
                    theta_vec = theta[parameter_config['theta']['selected_indices']]  # (T,)
                    theta = torch.from_numpy(theta_vec).unsqueeze(0).expand(phi.shape[0], -1)
                else:
                    theta = theta[:, parameter_config['theta']['selected_indices']]
                    theta = torch.from_numpy(theta)
            else:
                theta = torch.from_numpy(theta)

            # y / target
            tgt_ds = hdf[parameter_config['target']['key']]
            if tgt_ds.ndim > 1 and parameter_config['target']['selected_indices'] is not None:
                y = tgt_ds[:, parameter_config['target']['selected_indices']]
            else:
                y = tgt_ds[:].reshape(-1, 1)

        phi = torch.from_numpy(phi)
        y = torch.from_numpy(y)
        x = torch.cat([theta, phi], dim=-1)

    elif file_path.endswith('.csv'):
        # CSV via selected_labels
        df = pd.read_csv(file_path)

        def select_labels(df_: pd.DataFrame, labels: Union[str, Sequence[str]]) -> pd.DataFrame:
            if isinstance(labels, str):
                return df_[[labels]]
            elif isinstance(labels, (list, tuple)):
                return df_[list(labels)]
            else:
                raise ValueError(f"Invalid label type: {type(labels)}")

        phi_df = select_labels(df, parameter_config['phi']['selected_labels'])
        theta_df = select_labels(df, parameter_config['theta']['selected_labels'])
        y_df = select_labels(df, parameter_config['target']['selected_labels'])

        phi = torch.tensor(phi_df.values, dtype=torch.float32)
        theta = torch.tensor(theta_df.values, dtype=torch.float32)
        y = torch.tensor(y_df.values, dtype=torch.float32)

        if y.ndim == 1:
            y = y.unsqueeze(1)

        x = torch.cat([theta, phi], dim=-1)

    else:
        raise ValueError(f"Unsupported file format: {file_path}")

    # ensure float32 tensors
    x = x.contiguous().to(torch.float32)
    y = y.contiguous().to(torch.float32)
    return x, y


def _load_data_to_mem(files: Sequence[str], cfg: Dict) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    X, ys, file_inds = [], [], []
    for i, fp in enumerate(files):
        if not os.path.exists(fp):
            raise FileNotFoundError(fp)
        Xi, yi = _read_in_from_file(fp, cfg)
        X.append(Xi)
        ys.append(yi)
        file_inds.append(torch.full((Xi.size(0),), i, dtype=torch.long))
    x = torch.cat(X, 0).contiguous()
    y = torch.cat(ys, 0).contiguous()
    fidx = torch.cat(file_inds, 0).contiguous()
    return x, y, fidx





    




def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

In [3]:
def read_in_data():
        # Example parameter_config (edit to your keys/labels/indices)
        path_to_settings = "./binary-black-hole"
        with open(f"{path_to_settings}/settings.yaml", "r") as f:
            config_file = yaml.safe_load(f)
        sim = config_file["simulation_settings"]
        parameters = {
            "phi":    {"key": "phi",    "label_key": "phi_labels",    "selected_labels": sim["phi_labels"],    "size": len(sim["phi_labels"]),      "selected_indices": None},
            "theta":  {"key": "theta",  "label_key": "theta_headers", "selected_labels": sim["theta_labels"],  "size": len(sim["theta_labels"]),  "selected_indices": None},
            "target": {"key": "target", "label_key": "target_headers","selected_labels": sim["target_labels"], "size": len(sim["target_labels"]), "selected_indices": None},
        }
        files = _get_hdf5_files(Path(config_file["path_settings"]["path_to_files_train"]), config_file)

        if files[0].endswith(('.h5', '.hdf5')):
            parameters["phi"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["phi"])
            parameters["target"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["target"])
            parameters["theta"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["theta"])


        X_t, y_t, _ = _load_data_to_mem(files, parameters)
        # Ensure numpy arrays for kNN builder

        # X is your raw numpy or torch input, shape [N, D]
        #scaler = StandardScaler()

        # Convert to numpy if needed
        X_np = X_t.cpu().numpy() if isinstance(X_t, torch.Tensor) else X_t
        #X_t = X.detach().clone().to(torch.float32) if isinstance(X, torch.Tensor) else torch.tensor(X, dtype=torch.float32)

        # Fit on ALL data (or train split only)
        #X_np = scaler.fit_transform(X_np)

        # Back to torch
        X = X_np
        y_arr = y_t.cpu().numpy().reshape(-1, 1)

        # Ensure binary labels {0,1}. If multi-target or continuous, map/threshold here.
        if y_arr.shape[1] > 1:
            # choose a column or reduce to a binary indicator
            y = (y_arr[:, 0] > 0.5).astype(np.int64)
        else:
            if not np.array_equal(np.unique(y_arr), np.array([0, 1])):
                y = (y_arr[:, 0] > 0.5).astype(np.int64)
            else:
                y = y_arr[:, 0].astype(np.int64)
        
        X, y = shuffle(X, y, random_state=42)
        return X, y

In [4]:
# -------------------------
# Model components
# -------------------------
class SupConEncoder(nn.Module):
    def __init__(self, in_dim, enc_dim=256, proj_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(),
            nn.Linear(512, enc_dim), nn.ReLU(),
        )
        self.proj = nn.Sequential(
            nn.Linear(enc_dim, enc_dim), nn.ReLU(),
            nn.Linear(enc_dim, proj_dim),
        )
    def forward(self, x, *, return_proj=True):
        h = self.encoder(x)                # [B, enc_dim]
        if not return_proj:
            return h
        z = self.proj(h)                   # [B, proj_dim]
        z = F.normalize(z, dim=-1)         # cosine space
        return z

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature
    def forward(self, z, y):
        z = F.normalize(z, dim=-1)
        logits = (z @ z.t()) / self.tau
        logits = logits - torch.eye(z.size(0), device=z.device) * 1e9
        y = y.view(-1,1)
        pos_mask = (y == y.t()) & (y == 1)           # pull only positives
        log_denom = torch.logsumexp(logits, dim=1)
        num = torch.logsumexp(torch.where(pos_mask, logits, torch.full_like(logits, -1e9)), dim=1)
        valid = pos_mask.any(dim=1)
        loss = -(num[valid] - log_denom[valid]).mean()
        return loss

class MLPProbe(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, Z):                  # returns [N,1]
        return self.net(Z)

In [ ]:

# -------------------------
# Utilities
# -------------------------
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_label_aware_batches(y_np, batch_size=2048, min_pos=128, seed=0):
    rng = np.random.default_rng(seed)
    idx_pos = np.where(y_np==1)[0]
    idx_neg = np.where(y_np==0)[0]
    while True:
        p = rng.choice(idx_pos, size=min_pos, replace=(len(idx_pos) < min_pos))
        n = rng.choice(idx_neg, size=batch_size - min_pos, replace=False)
        b = np.concatenate([p, n]); rng.shuffle(b); yield b

@torch.no_grad()
def compute_z_embeddings(model, X_np, device="cpu", batch=65536):
    Z = []
    for i in range(0, len(X_np), batch):
        xb = torch.from_numpy(X_np[i:i+batch]).float().to(device)
        zb = model(xb, return_proj=True).cpu().numpy()
        Z.append(zb)
    Z = np.vstack(Z)
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    return Z

@torch.no_grad()
def logits_from_probe(head, Z, device="cpu"):
    out = head(torch.from_numpy(Z).float().to(device)).reshape(-1)
    return out.detach().cpu().numpy()

def zscore(a):
    a = np.asarray(a)
    return (a - a.mean()) / (a.std() + 1e-8)

def knn_density(Z_ref, Z_query, k=30):
    nn = NearestNeighbors(n_neighbors=k, metric='cosine').fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)
    sims = np.sum(Z_query[:,None,:] * Z_ref[idx], axis=2)  # mean cosine to k-NN
    return sims.mean(axis=1)

def topk_metrics(score, y, K=10):
    idx = np.argsort(-score)
    topk = y[idx][:K]
    return float(topk.mean()), float((topk.sum() > 0))

# -------------------------
# Training loops
# -------------------------
def train_supcon(X_tr, y_tr, in_dim, epochs=10, batch_size=2048, min_pos=128, lr=3e-4, wd=1e-4, tau=0.07, device="cpu"):
    model = SupConEncoder(in_dim=in_dim).to(device)
    crit = SupConLoss(temperature=tau)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sampler = make_label_aware_batches(y_tr, batch_size=batch_size, min_pos=min_pos, seed=0)
    steps = max(1, len(X_tr)//batch_size)
    model.train()
    for ep in range(epochs):
        run = 0.0
        for _ in range(steps):
            b = next(sampler)
            xb = torch.from_numpy(X_tr[b]).float().to(device)
            yb = torch.from_numpy(y_tr[b]).long().to(device)
            z = model(xb, return_proj=True)
            loss = crit(z, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run += loss.item()
        print(f"[SupCon] epoch {ep+1:02d} loss {run/steps:.4f}")
    model.eval()
    return model

def train_mlp_probe_on_Z(Z_tr, y_tr, epochs=10, lr=1e-3, wd=1e-4, device="cpu"):
    head = MLPProbe(Z_tr.shape[1]).to(device)
    pos_weight = torch.tensor([(y_tr==0).sum()/max(1,(y_tr==1).sum())], device=device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    Zt = torch.from_numpy(Z_tr).float().to(device)
    yt = torch.from_numpy(y_tr).float().to(device)
    head.train()
    for ep in range(epochs):
        opt.zero_grad()
        logits = head(Zt).reshape(-1)
        loss = F.binary_cross_entropy_with_logits(logits, yt, pos_weight=pos_weight)
        loss.backward(); opt.step()
    head.eval()
    return head

# -------------------------
# Two-stage scoring
# -------------------------
def blended_score(Z_ref, Z_query, head, alpha=0.2, k=30, device="cpu"):
    # Stage-A base: probe probability
    logits = logits_from_probe(head, Z_query, device=device)
    pr = 1.0 / (1.0 + np.exp(-logits))
    # Unsupervised density in z-space
    dens = knn_density(Z_ref, Z_query, k=k)
    # z-score blend
    s = alpha * zscore(pr) + (1.0 - alpha) * zscore(dens)
    return s

def evaluate_all(Z_tr, y_tr, Z_va, y_va, Z_te, y_te, head, alpha=0.2, k=30, M=100, device="cpu"):
    # Global AP/ROC using probe (Stage A)
    pr_va = 1/(1+np.exp(-logits_from_probe(head, Z_va, device=device)))
    pr_te = 1/(1+np.exp(-logits_from_probe(head, Z_te, device=device)))
    ap_va = average_precision_score(y_va, pr_va); roc_va = roc_auc_score(y_va, pr_va)
    ap_te = average_precision_score(y_te, pr_te); roc_te = roc_auc_score(y_te, pr_te)

    # Local re-rank score (Stage B) for top-K metrics directly on whole set (or use top-M)
    s_va = blended_score(Z_tr, Z_va, head, alpha=alpha, k=k, device=device)
    s_te = blended_score(Z_tr, Z_te, head, alpha=alpha, k=k, device=device)
    pur_va, hit_va = topk_metrics(s_va, y_va, K=10)
    pur_te, hit_te = topk_metrics(s_te, y_te, K=10)

    print(f"FINAL VAL  PR-AUC={ap_va:.4f} | ROC-AUC={roc_va:.4f} | purity@10={pur_va:.3f} | P(≥1@10)={hit_va:.3f}")
    print(f"FINAL TEST PR-AUC={ap_te:.4f} | ROC-AUC={roc_te:.4f} | purity@10={pur_te:.3f} | P(≥1@10)={hit_te:.3f}")
    return (ap_va, roc_va, pur_va, hit_va), (ap_te, roc_te, pur_te, hit_te)





In [12]:
def ranknorm(x):
    x = np.asarray(x); r = np.argsort(np.argsort(x))
    return r / (len(x)-1 + 1e-9)

def ranknorm(x):
    x = np.asarray(x); r = np.argsort(np.argsort(x))
    return r / (len(x) - 1 + 1e-9)

@torch.no_grad()
def probe_prob(head, Z, device="cpu"):
    logits = head(torch.from_numpy(Z).float().to(device)).reshape(-1).cpu().numpy()
    return 1.0 / (1.0 + np.exp(-logits))

def knn_density(Z_ref, Z_query, k=20):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine").fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)
    sims = np.sum(Z_query[:,None,:] * Z_ref[idx], axis=2)  # mean cosine to k-NN
    return sims.mean(axis=1)

def blended_rank_score(Z_ref, Z_query, head, alpha=0.5, k=20, device="cpu"):
    pr  = probe_prob(head, Z_query, device=device)
    den = knn_density(Z_ref, Z_query, k=k)
    return alpha * ranknorm(pr) + (1.0 - alpha) * ranknorm(den)

def rerank_topM(Z_ref, Z_query, head, M=200, alpha=0.5, k=20, device="cpu", K=10):
    pr = probe_prob(head, Z_query, device=device)
    idxM = np.argpartition(-pr, M)[:M]
    sM = blended_rank_score(Z_ref, Z_query[idxM], head, alpha=alpha, k=k, device=device)
    return idxM[np.argsort(-sM)[:K]]

In [13]:
def rerank_topK(
    Z_ref,
    Z_query,
    head,
    M=200,
    K=10,
    alpha=0.5,
    k=20,
    device="cpu"
):
    """
    1) Use probe to select top-M candidates
    2) Re-rank those M using blended score
    3) Return final top-K indices (relative to full Z_query)

    Returns:
        np.ndarray of indices, shape [K]
    """

    # -------------- Stage 1: retrieve top-M by probe --------------
    pr = probe_prob(head, Z_query, device=device)
    idxM = np.argpartition(-pr, M)[:M]

    # -------------- Stage 2: re-rank top-M by blended score -------
    scores = blended_rank_score(
        Z_ref,
        Z_query[idxM],
        head,
        alpha=alpha,
        k=k,
        device=device
    )

    reranked = idxM[np.argsort(-scores)[:K]]
    return reranked

In [18]:
# -------------------------
# End-to-end runner
# -------------------------
def run_pipeline(X_np, y_np, *, test_size=0.2, val_size=0.2, seed=42,
                 supcon_epochs=10, supcon_tau=0.07, supcon_batch=1024, supcon_min_pos=128,
                 probe_epochs=10, device="cpu",
                 alpha=0.2, k_density=30):
    """
    X_np: raw features [N,D], y_np: {0,1}
    """
    set_seed(seed)

    # Split (train->val/test)
    X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=test_size, stratify=y_np, random_state=seed)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=val_size, stratify=y_tr, random_state=seed)
    print("train pos:", int(y_tr.sum()), "val pos:", int(y_va.sum()), "test pos:", int(y_te.sum()))
    # Fit scaler on TRAIN only
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr); X_va = scaler.transform(X_va); X_te = scaler.transform(X_te)

    # SupCon pretrain on TRAIN
    model = train_supcon(
        X_tr, y_tr, in_dim=X_tr.shape[1],
        epochs=supcon_epochs, batch_size=supcon_batch, min_pos=supcon_min_pos,
        lr=3e-4, wd=1e-4, tau=supcon_tau, device=device
    )

    # z-embeddings
    Z_tr = compute_z_embeddings(model, X_tr, device=device)
    Z_va = compute_z_embeddings(model, X_va, device=device)
    Z_te = compute_z_embeddings(model, X_te, device=device)

    # MLP probe on z (TRAIN only)
    head = train_mlp_probe_on_Z(Z_tr, y_tr, epochs=probe_epochs, lr=1e-3, wd=1e-4, device=device)

    import numpy as np
    # 1) sanity: alignments & pathologies
    assert len(y_va) == len(Z_va)
    s_va = blended_score(Z_tr, Z_va, head, alpha=0.2, k=30, device=device)
    print("len", len(s_va), "nan", np.isnan(s_va).sum(), "inf", np.isinf(s_va).sum(),
        "min/max", np.nanmin(s_va), np.nanmax(s_va))

    # 2) are we sorting the right way?
    idx = np.argsort(-s_va)
    print("top10 labels:", y_va[idx][:10], "sum:", int(y_va[idx][:10].sum()))

    # 3) compare probe-only vs blend: if probe works but blend fails, it’s α/scale
    pr_va = 1/(1+np.exp(-logits_from_probe(head, Z_va, device=device)))
    idxp = np.argsort(-pr_va)
    print("probe top10 labels:", y_va[idxp][:10], "sum:", int(y_va[idxp][:10].sum()))
    
    # 4) check score scales/correlation
    def zscore(a): a=np.asarray(a); return (a - a.mean())/(a.std()+1e-8)
    

    """
    # cache probe probs & density features
    pr_va = 1/(1+np.exp(-logits_from_probe(head, Z_va, device=device)))
    pr_te = 1/(1+np.exp(-logits_from_probe(head, Z_te, device=device)))

    def knn_density(Z_ref, Z_query, k):
        nn = NearestNeighbors(n_neighbors=k, metric='cosine').fit(Z_ref)
        _, idx = nn.kneighbors(Z_query)
        sims = np.sum(Z_query[:,None,:] * Z_ref[idx], axis=2)
        return sims.mean(axis=1)
    
    best = None
    for kden in (20,30,40,60):
        dens_va = knn_density(Z_tr, Z_va, k=kden)
        dens_te = knn_density(Z_tr, Z_te, k=kden)
        pr_va_z, dens_va_z = zscore(pr_va), zscore(dens_va)
        pr_te_z, dens_te_z = zscore(pr_te), zscore(dens_te)
        for alpha in (0.1,0.2,0.3,0.4,0.5,0.6):
            s_va = alpha*pr_va_z + (1-alpha)*dens_va_z
            pur_va, hit_va = topk_metrics(s_va, y_va, K=10)
            if (best is None) or (pur_va, hit_va) > (best[0], best[1]):  # lexicographic: purity then hit
                best = (pur_va, hit_va, alpha, kden, pr_te_z, dens_te_z)
                s_best_va = s_va
    print("BEST on VAL:", {"purity":best[0], "P>=1":best[1], "alpha":best[2], "k":best[3]})

    dens_va = knn_density(Z_tr, Z_va, k=best[3])
    from scipy.stats import spearmanr
    print("rho(pr,dens):", spearmanr(pr_va, dens_va).correlation)
    print("zstd(pr_z,dens_z):", zscore(pr_va).std(), zscore(dens_va).std())
    
    # apply to TEST
    alpha, kden, pr_te_z, dens_te_z = best[2], best[3], best[4], best[5]
    s_te = alpha*pr_te_z + (1-alpha)*dens_te_z
    pur_te, hit_te = topk_metrics(s_te, y_te, K=10)
    print("TEST purity@10=%.3f | P(>=1@10)=%.3f" % (pur_te, hit_te))
    """
    # Global (probe only) – this is what you report for AP/ROC
    pr_va = probe_prob(head, Z_va, device=device)
    pr_te = probe_prob(head, Z_te, device=device)
    from sklearn.metrics import average_precision_score, roc_auc_score
    print("VAL  AP/ROC:", average_precision_score(y_va, pr_va), roc_auc_score(y_va, pr_va))
    print("TEST AP/ROC:", average_precision_score(y_te, pr_te), roc_auc_score(y_te, pr_te))

    # Top-K (blended rank score) – this is what you use for purity@10
    s_va = blended_rank_score(Z_tr, Z_va, head, alpha=0.5, k=20, device=device)
    s_te = blended_rank_score(Z_tr, Z_te, head, alpha=0.5, k=20, device=device)

    def topk_metrics(score, y, K=10):
        idx = np.argsort(-score)[:K]
        return float(y[idx].mean()), float((y[idx].sum() > 0))

    print("VAL  purity@10, P>=1:", topk_metrics(s_va, y_va))
    print("TEST purity@10, P>=1:", topk_metrics(s_te, y_te))

    def rerank_topK(Z_ref, Z_query, head, M=200, K=10, alpha=0.5, k=20, device="cpu"):
        pr = probe_prob(head, Z_query, device=device)
        idxM = np.argpartition(-pr, M)[:M]
        sM = blended_rank_score(Z_ref, Z_query[idxM], head, alpha=alpha, k=k, device=device)
        return idxM[np.argsort(-sM)[:K]]

    top_va = rerank_topK(Z_tr, Z_va, head, M=200, K=10, alpha=0.5, k=20, device=device)
    top_te = rerank_topK(Z_tr, Z_te, head, M=200, K=10, alpha=0.5, k=20, device=device)
    print("VAL  purity@10, P>=1:", float(y_va[top_va].mean()), float((y_va[top_va].sum()>0)))
    print("TEST purity@10, P>=1:", float(y_te[top_te].mean()), float((y_te[top_te].sum()>0)))



    # Evaluate (Stage-A global + Stage-B blended)
    return evaluate_all(Z_tr, y_tr, Z_va, y_va, Z_te, y_te, head, alpha=alpha, k=k_density, device=device)

In [19]:
# -------------------------
# Example (replace with your loader)
# -------------------------
from os import read


if __name__ == "__main__":
    # Toy usage: replace X_np, y_np with your data loader
    # X_np: (N,D) float32; y_np: (N,) int64 in {0,1}
    X_np, y_np = read_in_data()
    X_np = X_np[:100000,:]
    y_np = y_np[:100000]

    run_pipeline(
        X_np, y_np,
        supcon_epochs=10, supcon_tau=0.07, supcon_batch=1024, supcon_min_pos=128,
        probe_epochs=10, device="cpu",
        alpha=0.5, k_density=20
    )

train pos: 453 val pos: 113 test pos: 142
[SupCon] epoch 01 loss 0.4759
[SupCon] epoch 02 loss 0.1794
[SupCon] epoch 03 loss 0.1171
[SupCon] epoch 04 loss 0.0931
[SupCon] epoch 05 loss 0.0768
[SupCon] epoch 06 loss 0.0651
[SupCon] epoch 07 loss 0.0478
[SupCon] epoch 08 loss 0.0415
[SupCon] epoch 09 loss 0.0418
[SupCon] epoch 10 loss 0.0418
len 16000 nan 0 inf 0 min/max -3.6694033 0.8553022
top10 labels: [0 0 0 0 0 0 0 0 0 0] sum: 0
probe top10 labels: [0 1 0 1 0 0 0 0 0 1] sum: 3
VAL  AP/ROC: 0.14206628637634458 0.9513121152653893
TEST AP/ROC: 0.11902731932531396 0.9504410894817997
VAL  purity@10, P>=1: (0.0, 0.0)
TEST purity@10, P>=1: (0.0, 0.0)
VAL  purity@10, P>=1: 0.4 1.0
TEST purity@10, P>=1: 0.5 1.0
VAL  PR-AUC=0.1421 | ROC-AUC=0.9513 | purity@10=0.400 | P(≥1@10)=1.000
TEST PR-AUC=0.1190 | ROC-AUC=0.9504 | purity@10=0.500 | P(≥1@10)=1.000
